# Dividend based investing  @Mayank_Srivastav


In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import math
from scipy import stats

In [2]:
tickers = pd.read_csv('top_50_indian_stocks.csv')
tickers.head()

,Ticker,Company Name
0,RELIANCE.NS,Reliance Industries
1,TCS.NS,Tata Consultancy Services
2,HDFCBANK.NS,HDFC Bank
3,INFY.NS,Infosys
4,ICICIBANK.NS,ICICI Bank


Weighted Scoring Model for finding the final score 

In [9]:
def create_dividend_df(tickers):
    columns = [
        "Ticker", 
        "Dividend Yield(%)", 
        "Dividend Rate",
        "Payout Ratio(%)", 
        "5 Year Avg Dividend Yield(%)", 
        "Earning Growth(%)"
        
    ]
    
    dividend_df = pd.DataFrame(columns=columns)
    
    for stock in tickers:
        ticker = yf.Ticker(stock)
        info = ticker.info 
        
        dividend_yield = info.get("dividendYield", np.nan) * 100 if info.get("dividendYield") else np.nan
        dividend_rate = info.get("dividendRate", np.nan) 
        payout_ratio = info.get("payoutRatio", np.nan) * 100 if info.get("payoutRatio") else np.nan
        five_year_avg_dividend_yield = info.get("fiveYearAvgDividendYield", np.nan) * 100 if info.get("fiveYearAvgDividendYield") else np.nan
        earning_growth = info.get("earningsGrowth", np.nan) * 100 if info.get("earningsGrowth") else np.nan
        
        dividend_df.loc[len(dividend_df)] = [stock, dividend_yield, dividend_rate, payout_ratio, five_year_avg_dividend_yield, earning_growth]
        
    
    numeric_cols = [
          "Dividend Yield(%)", 
        "Dividend Rate",
        "Payout Ratio(%)", 
        "5 Year Avg Dividend Yield(%)", 
        "Earning Growth(%)"
    ]
    

    for col in numeric_cols:
        if col == "Payout Ratio":
            dividend_df[col+ " Normalised"] = 1 - ((dividend_df[col] - dividend_df[col].min()) / (dividend_df[col].max() - dividend_df[col].min()))
        else:
            dividend_df[col+ " Normalised"] = (dividend_df[col] - dividend_df[col].min()) / (dividend_df[col].max() - dividend_df[col].min())
        
    return dividend_df

In [4]:
tickers_list = tickers["Ticker"].values.tolist()
dividend_df = create_dividend_df(tickers_list)
dividend_df

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised
0,RELIANCE.NS,46.0,6.0,9.21,44.0,-12.6,0.043433,0.034483,0.085562,0.047146,0.068275
1,TCS.NS,592.0,124.0,46.32,149.0,12.2,0.608066,0.774295,0.48098,0.177419,0.09128
2,HDFCBANK.NS,163.0,13.0,21.76,102.0,7.5,0.164426,0.07837,0.219286,0.119107,0.08692
3,INFY.NS,480.0,50.0,64.65,240.0,11.8,0.492244,0.310345,0.676292,0.290323,0.090909
4,ICICIBANK.NS,79.0,11.0,14.71,99.0,8.4,0.077559,0.065831,0.144166,0.115385,0.087755
5,HINDUNILVR.NS,202.0,44.0,95.03,158.0,21.4,0.204757,0.272727,1.0,0.188586,0.099814
6,SBIN.NS,166.0,17.35,17.44,150.0,-3.1,0.167528,0.105643,0.173255,0.17866,0.077087
7,BAJFINANCE.NS,55.0,5.4,14.42,24.0,21.4,0.05274,0.030721,0.141076,0.022333,0.099814
8,BHARTIARTL.NS,86.0,16.0,36.060002,80.0,-34.0,0.084798,0.097179,0.371657,0.091811,0.048423
9,ITC.NS,552.0,16.0,86.92,372.0,-72.7,0.566701,0.097179,0.913586,0.454094,0.012523


In [5]:
weights = {
        "Dividend Yield(%) Normalised": 0.3,
        "Dividend Rate Normalised": 0.2,
        "Payout Ratio(%) Normalised": 0.2,
        "5 Year Avg Dividend Yield(%) Normalised": 0.2,
        "Earning Growth(%) Normalised": 0.1
    }

dividend_df["Dividend Score"] = dividend_df[[col for col in weights.keys()]].mul(list(weights.values())).sum(axis = 1)
dividend_df

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised,Dividend Score
0,RELIANCE.NS,46.0,6.0,9.21,44.0,-12.6,0.043433,0.034483,0.085562,0.047146,0.068275,0.053296
1,TCS.NS,592.0,124.0,46.32,149.0,12.2,0.608066,0.774295,0.48098,0.177419,0.09128,0.478087
2,HDFCBANK.NS,163.0,13.0,21.76,102.0,7.5,0.164426,0.07837,0.219286,0.119107,0.08692,0.141372
3,INFY.NS,480.0,50.0,64.65,240.0,11.8,0.492244,0.310345,0.676292,0.290323,0.090909,0.412156
4,ICICIBANK.NS,79.0,11.0,14.71,99.0,8.4,0.077559,0.065831,0.144166,0.115385,0.087755,0.09712
5,HINDUNILVR.NS,202.0,44.0,95.03,158.0,21.4,0.204757,0.272727,1.0,0.188586,0.099814,0.363671
6,SBIN.NS,166.0,17.35,17.44,150.0,-3.1,0.167528,0.105643,0.173255,0.17866,0.077087,0.149479
7,BAJFINANCE.NS,55.0,5.4,14.42,24.0,21.4,0.05274,0.030721,0.141076,0.022333,0.099814,0.06463
8,BHARTIARTL.NS,86.0,16.0,36.060002,80.0,-34.0,0.084798,0.097179,0.371657,0.091811,0.048423,0.142411
9,ITC.NS,552.0,16.0,86.92,372.0,-72.7,0.566701,0.097179,0.913586,0.454094,0.012523,0.464234


Sort according to Dividend score 

In [ ]:
dividend_df = dividend_df[dividend_df["Dividend Yield(%)"] > 0].copy()

dividend_df = dividend_df.sort_values(by = "Dividend Score", ascending = False)
dividend_df.head(10)

NameError: name 'df' is not defined